# 🚀 Nhiệm vụ 07.1, 07.2 & 07.3: Dự báo Sales Nâng cao
**Ngày 4:** Xây dựng mô hình, Giải thích với SHAP và Xuất kết quả nộp bài.
**Team:** Outliers

## 1. Khai báo Thư viện & Cấu hình

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import plotly.graph_objects as go
import plotly.express as px
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

## 2. Tải và Chuẩn bị Dữ liệu

In [ ]:
data_path = "../Data/processed_train.csv"
if not os.path.exists(data_path): data_path = "Data/processed_train.csv"

df = pl.read_csv(data_path).with_columns(pl.col("Date").str.to_date()).sort("Date")
df_pd = df.to_pandas()
print(f"Tổng số bản ghi: {len(df_pd)}")

## 3. Lựa chọn Đặc trưng & Huấn luyện (Task 07.1)

In [ ]:
features = [
    'month', 'day', 'dayofweek', 'is_weekend', 'is_payday', 'is_double_day',
    'traffic', 'is_promo', 'total_stock', 'total_units_sold_lag1',
    'revenue_lag_1', 'revenue_lag_7', 'revenue_lag_14', 'revenue_lag_30',
    'revenue_roll_mean_7', 'revenue_roll_mean_30',
    'sin_year', 'cos_year', 'is_pre_tet', 'is_tet_week', 'is_post_tet'
]
target = 'Revenue'

train_data = df_pd[df_pd['Date'].dt.year < 2022]
val_data = df_pd[df_pd['Date'].dt.year == 2022]

X_train, y_train = train_data[features], train_data[target]
X_val, y_val = val_data[features], val_data[target]

model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print("Đã huấn luyện xong mô hình XGBoost.")

## 4. Biểu đồ Động & Đánh giá

In [ ]:
y_pred = model.predict(X_val)
mae = mean_absolute_error(y_val, y_pred)

fig = go.Figure()
fig.add_trace(go.Scatter(x=val_data['Date'], y=y_val, name='Thực tế'))
fig.add_trace(go.Scatter(x=val_data['Date'], y=y_pred, name='Dự báo', line=dict(dash='dot')))
fig.update_layout(title=f'Dự báo vs Thực tế 2022 (MAE: {mae:,.0f})', width=1400, height=600, template='plotly_white')
fig.show()

## 5. Giải thích mô hình chuyên sâu với SHAP (Task 07.2)
SHAP giúp chúng ta hiểu rõ đóng góp của từng tính năng vào dự báo cuối cùng.

In [ ]:
# Khởi tạo SHAP Explainer (Sử dụng mẫu 100 bản ghi để tăng tốc độ tính toán)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_val.sample(200, random_state=42))

# 5.1 SHAP Summary Plot
print("Đang tính toán SHAP Summary Plot...")
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_val.sample(200, random_state=42), plot_type="dot")

# 5.2 SHAP Bar Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_val.sample(200, random_state=42), plot_type="bar")

### Phân tích từ SHAP:
1. **Màu sắc (Đỏ = Giá trị cao, Xanh = Giá trị thấp)**: Nếu các điểm đỏ nằm về phía bên phải (SHAP value dương), nghĩa là tính năng đó khi tăng lên sẽ làm tăng doanh thu.
2. **Revenue_lag_1**: Có dải điểm đỏ rộng bên phải, xác nhận ngày hôm qua cao thì hôm nay có xu hướng cao.
3. **Is_double_day**: Các điểm đỏ (giá trị 1) đẩy doanh thu lên rất mạnh.

## 6. Xuất kết quả nộp bài (Task 07.3)

In [ ]:
# Huấn luyện cuối cùng
X_all = df_pd[features]
y_all_rev = df_pd['Revenue']
y_all_cogs = df_pd['COGS']

final_model_rev = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42)
final_model_rev.fit(X_all, y_all_rev)

final_model_cogs = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42)
final_model_cogs.fit(X_all, y_all_cogs)

sub_path = "../Data/sample_submission.csv"
if not os.path.exists(sub_path): sub_path = "Data/sample_submission.csv"
submission = pd.read_csv(sub_path)
submission['Date'] = pd.to_datetime(submission['Date'])

test_results = []
current_data = df_pd.copy()

for i, row in submission.iterrows():
    d = row['Date']
    feat = {
        'month': d.month, 'day': d.day, 'dayofweek': d.dayofweek,
        'is_weekend': 1 if d.dayofweek >= 5 else 0,
        'is_payday': 1 if d.day in [15, 30, 31] else 0,
        'is_double_day': 1 if d.month == d.day else 0,
        'traffic': df_pd['traffic'].mean(), 'is_promo': 0,
        'total_stock': df_pd['total_stock'].mean(), 'total_units_sold_lag1': df_pd['total_units_sold_lag1'].mean(),
        'sin_year': np.sin(2 * np.pi * d.dayofyear / 365.25), 'cos_year': np.cos(2 * np.pi * d.dayofyear / 365.25),
        'revenue_lag_1': current_data['Revenue'].iloc[-1],
        'revenue_lag_7': current_data['Revenue'].iloc[-7],
        'revenue_lag_14': current_data['Revenue'].iloc[-14],
        'revenue_lag_30': current_data['Revenue'].iloc[-30],
        'revenue_roll_mean_7': current_data['Revenue'].tail(7).mean(),
        'revenue_roll_mean_30': current_data['Revenue'].tail(30).mean(),
        'is_pre_tet': 0, 'is_tet_week': 0, 'is_post_tet': 0
    }
    X_test = pd.DataFrame([feat])[features]
    p_rev = final_model_rev.predict(X_test)[0]
    p_cogs = final_model_cogs.predict(X_test)[0]
    test_results.append({'Date': d, 'Revenue': p_rev, 'COGS': p_cogs})
    current_data = pd.concat([current_data, pd.DataFrame([{'Revenue': p_rev}])], ignore_index=True)

final_sub = pd.DataFrame(test_results)
final_sub.to_csv("submission_v2_advanced_final.csv", index=False)
print("✅ Đã tạo thành công: submission_v2_advanced_final.csv")